# Image Processing: Noise, Filtering and Histograms

This notebook loads `usthb.jpg`, converts it to grayscale, adds Gaussian noise, applies smoothing filters, and compares image histograms before and after intensity transformations and histogram equalization.

**Before running:** put `usthb.jpg` in the same folder as this notebook.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Load the image and convert it to grayscale

In [ ]:
img = cv2.imread("usthb.jpg", cv2.IMREAD_COLOR)

if img is None:
    raise FileNotFoundError("Image could not be loaded. Put usthb.jpg next to the notebook or update the path.")

# OpenCV loads color images in BGR format; convert to grayscale.
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(7, 5))
plt.imshow(gray, cmap='gray', vmin=0, vmax=255)
plt.title('Grayscale Image')
plt.axis('off')
plt.show()

## 2. Generate and add Gaussian noise

In [ ]:
# Generate Gaussian noise with mean 0 and standard deviation 25.
noise = np.random.normal(loc=0, scale=25, size=gray.shape)

# Add the noise using float values, then clip to the valid pixel range.
noisy = gray.astype(np.float32) + noise
noisy = np.clip(noisy, 0, 255).astype(np.uint8)

# Normalize the noise only to make it visible as an image.
noise_display = cv2.normalize(noise, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original grayscale')
axes[1].imshow(noise_display, cmap='gray')
axes[1].set_title('Generated noise (display)')
axes[2].imshow(noisy, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Noisy image')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Apply smoothing filters

The kernel size must be a positive odd number. The helper below converts a slider value into an odd kernel size using `2 * value + 1`.

In [ ]:
def show_filters(value=2):
    k = 2 * value + 1

    mean_filtered = cv2.blur(noisy, (k, k))
    gaussian_filtered = cv2.GaussianBlur(noisy, (k, k), 0)
    median_filtered = cv2.medianBlur(noisy, k)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    images = [noisy, mean_filtered, gaussian_filtered, median_filtered]
    titles = [
        'Noisy image',
        f'Mean filter ({k}×{k})',
        f'Gaussian filter ({k}×{k})',
        f'Median filter ({k}×{k})'
    ]

    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_filters(2)  # Kernel size = 5 × 5

### Optional: interactive kernel-size slider

Run this cell if `ipywidgets` is installed. It replaces the OpenCV trackbars, which do not work reliably inside Jupyter Notebook.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    slider = widgets.IntSlider(value=2, min=1, max=20, step=1, description='Kernel value:')
    display(widgets.interactive(show_filters, value=slider))
except ImportError:
    print('ipywidgets is not installed. Run: %pip install ipywidgets')

## 4. Histograms after intensity transformations

In [ ]:
# Calculate the original grayscale histogram.
hist = cv2.calcHist([gray], [0], None, [256], [0, 256])

# Divide intensities by 2, reducing the range to approximately 0–127.
gray2 = (gray / 2).astype(np.uint8)

# Divide by 2 and shift intensities upward by 127.
gray3 = (gray / 2 + 127).astype(np.uint8)

hist2 = cv2.calcHist([gray2], [0], None, [256], [0, 256])
hist3 = cv2.calcHist([gray3], [0], None, [256], [0, 256])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original grayscale')
axes[0].axis('off')
axes[1].imshow(gray2, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Intensity divided by 2')
axes[1].axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(hist, label='Original')
plt.plot(hist2, label='Divided by 2')
plt.plot(hist3, label='Divided by 2 + 127')
plt.title('Histograms after intensity transformations')
plt.xlabel('Pixel intensity')
plt.ylabel('Number of pixels')
plt.xlim([0, 256])
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 5. Histogram equalization

In [ ]:
# gray is already an 8-bit image (0–255), so no multiplication by 255 is needed.
gray_eq = cv2.equalizeHist(gray)

hist_original = cv2.calcHist([gray], [0], None, [256], [0, 256])
hist_equalized = cv2.calcHist([gray_eq], [0], None, [256], [0, 256])

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original grayscale image')
axes[0, 1].imshow(gray_eq, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('Histogram-equalized image')
axes[1, 0].plot(hist_original)
axes[1, 0].set_title('Original histogram')
axes[1, 1].plot(hist_equalized)
axes[1, 1].set_title('Equalized histogram')

for ax in axes[0, :]:
    ax.axis('off')
for ax in axes[1, :]:
    ax.set_xlabel('Pixel intensity')
    ax.set_ylabel('Number of pixels')
    ax.set_xlim([0, 256])
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 6. Conclusion

- Gaussian noise adds random intensity variations to the grayscale image.
- Mean, Gaussian, and median filters smooth the noisy image in different ways.
- Intensity transformations change the distribution of pixel values and therefore the histogram.
- Histogram equalization redistributes grayscale intensities to improve contrast in many images.